# StreamingAttention — Sprint 1: Zero-Shot Validation

**Goal**: Load Llama-3.1-8B, classify heads (DuoAttention patterns or entropy), replace streaming heads with recurrent state, measure perplexity degradation.

**GO/NO-GO**: Zero-shot perplexity < 3x baseline → proceed to calibration. > 10x → re-evaluate.

---

## Cell 1: Environment Setup

In [1]:
# === Colab Setup ===
!pip install -q flash-linear-attention transformers datasets peft accelerate matplotlib tqdm

import os
if not os.path.exists('PROJECT'):
    !git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
%cd PROJECT
!pip install -e .

# HuggingFace login (required for gated models like Llama)
from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    print('Set HF_TOKEN in Colab secrets, or run: login(token="hf_...")')

# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/streaming_attention_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Verify streaming_attention is importable
import streaming_attention
print(f"streaming_attention v{streaming_attention.__version__} loaded successfully")

import json
import logging
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('streaming_attention')

# Check GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Reset memory stats for tracking
    torch.cuda.reset_peak_memory_stats()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.3/437.3 kB 44.4 MB/s eta 0:00:00
Cloning into 'PROJECT'...
remote: Enumerating objects: 2232, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 2232 (delta 55), reused 0 (delta 0), pack-reused 2097 (from 1)
Receiving objects: 100% (2232/2232), 16.67 MiB | 4.07 MiB/s, done.
Resolving deltas: 100% (1718/1718), done.
/content/PROJECT
Obtaining file:///content/PROJECT
  Preparing metadata (setup.py) ... done
  Running setup.py develop for streaming_attention
Mounted at /content/drive
streaming_attention v0.1.0 loaded successfully
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
Using device: cuda


## Cell 2: Load Llama-3.1-8B-Instruct

In [2]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {MODEL_NAME} (bfloat16)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',  # Need eager attention to intercept forward pass
)
model.eval()

# Print model config
cfg = model.config
print(f"\nModel config:")
print(f"  Layers: {cfg.num_hidden_layers}")
print(f"  Q heads: {cfg.num_attention_heads}")
print(f"  KV heads: {cfg.num_key_value_heads}")
print(f"  Head dim: {cfg.hidden_size // cfg.num_attention_heads}")
print(f"  Hidden size: {cfg.hidden_size}")
print(f"  GQA ratio: {cfg.num_attention_heads // cfg.num_key_value_heads} Q per KV head")

param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"  Model memory: {param_bytes / 1e9:.2f} GB")
if torch.cuda.is_available():
    print(f"  GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading tokenizer: meta-llama/Llama-3.1-8B-Instruct


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Model config:
  Layers: 32
  Q heads: 32
  KV heads: 8
  Head dim: 128
  Hidden size: 4096
  GQA ratio: 4 Q per KV head
  Model memory: 16.06 GB
  GPU memory allocated: 16.06 GB


## Cell 3: Baseline Perplexity (Full KV Cache)

In [3]:
from streaming_attention.eval_perplexity import evaluate_perplexity

print("="*60)
print("BASELINE: Full KV cache perplexity on WikiText-2")
print("="*60)

baseline_result = evaluate_perplexity(
    model=model,
    tokenizer=tokenizer,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_length=2048,
    stride=512,
    device=DEVICE,
)

print(f"\n>>> Baseline perplexity: {baseline_result['perplexity']:.2f}")
print(f">>> Average loss: {baseline_result['avg_loss']:.4f}")
print(f">>> Tokens evaluated: {baseline_result['num_tokens']}")

if torch.cuda.is_available():
    print(f">>> Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    torch.cuda.reset_peak_memory_stats()

BASELINE: Full KV cache perplexity on WikiText-2


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (289077 > 131072). Running this sequence through the model will result in indexing errors

Perplexity (wikitext):  99%|█████████▉| 561/565 [02:34<00:01,  3.62it/s, ppl=6.43, tokens=288768]


>>> Baseline perplexity: 6.43
>>> Average loss: 1.8607
>>> Tokens evaluated: 289077
>>> Peak GPU memory: 19.24 GB


## Cell 4: Head Classification

Two options:
- **Option A**: Load DuoAttention pre-computed patterns (preferred, faster)
- **Option B**: Compute attention entropy from scratch (fallback)

In [4]:
from streaming_attention.head_classifier import load_duo_attention_patterns, compute_attention_entropy, HeadClassification

# =============================================================================
# STREAMING_FRACTION: Controls what fraction of heads get converted to state.
# Lower = safer (fewer heads converted). Start with 0.2 for zero-shot.
# =============================================================================
STREAMING_FRACTION = 0.5

# --- Option A: DuoAttention patterns (PREFERRED) ---
if not os.path.exists('duo-attention'):
    print("Cloning DuoAttention for pre-computed head patterns...")
    !git clone --depth 1 https://github.com/mit-han-lab/duo-attention.git duo-attention

DUO_ATTENTION_DIR = 'duo-attention/attn_patterns/Meta-Llama-3.1-8B-Instruct'

if os.path.exists(DUO_ATTENTION_DIR):
    print(f"Loading DuoAttention patterns (streaming fraction={STREAMING_FRACTION:.0%})...")
    head_classification = load_duo_attention_patterns(
        DUO_ATTENTION_DIR,
        sparsity=STREAMING_FRACTION,
    )
    classification_method = "DuoAttention"

# --- Option B: Entropy-based classification (fallback) ---
else:
    print("DuoAttention patterns not found. Using entropy-based classification...")
    print("(This requires a forward pass with output_attentions=True)")

    from datasets import load_dataset
    from torch.utils.data import DataLoader

    entropy_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    entropy_dataset = entropy_dataset.filter(lambda x: len(x['text'].strip()) > 50)

    def tokenize_fn(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=512,
            padding='max_length',
            return_tensors='pt',
        )

    entropy_dataset = entropy_dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
    entropy_dataset.set_format('torch')
    entropy_loader = DataLoader(entropy_dataset, batch_size=2, shuffle=False)

    head_classification = compute_attention_entropy(
        model=model,
        dataloader=entropy_loader,
        max_batches=20,
        device=DEVICE,
    )

    # Override Otsu threshold with target streaming fraction
    if head_classification.raw_scores is not None:
        scores = head_classification.raw_scores
        threshold = torch.quantile(scores.flatten(), 1.0 - STREAMING_FRACTION).item()
        new_mask = scores < threshold  # Low entropy = retrieval
        head_classification = HeadClassification(
            mask=new_mask,
            num_retrieval=int(new_mask.sum().item()),
            num_streaming=int((~new_mask).sum().item()),
            raw_scores=scores,
        )
        print(f"Overrode Otsu threshold → streaming fraction = {STREAMING_FRACTION:.0%}")

    classification_method = "Entropy"

# --- Summary ---
print(f"\nClassification method: {classification_method}")
print(f"Mask shape: {head_classification.mask.shape} (layers x KV heads)")
print(f"Retrieval heads: {head_classification.num_retrieval}")
print(f"Streaming heads: {head_classification.num_streaming}")
print(f"Streaming fraction: {head_classification.streaming_fraction:.1%}")

# Show per-layer breakdown
mask = head_classification.mask
print(f"\nPer-layer breakdown (R=retrieval, S=streaming):")
for layer_idx in range(mask.shape[0]):
    heads = ''.join(['R' if mask[layer_idx, h] else 'S' for h in range(mask.shape[1])])
    n_streaming = (~mask[layer_idx]).sum().item()
    print(f"  Layer {layer_idx:2d}: [{heads}] ({n_streaming}/{mask.shape[1]} streaming)")

Cloning DuoAttention for pre-computed head patterns...
Cloning into 'duo-attention'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 146 (delta 14), reused 135 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 15.10 MiB | 12.74 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Loading DuoAttention patterns (streaming fraction=50%)...

Classification method: DuoAttention
Mask shape: torch.Size([32, 8]) (layers x KV heads)
Retrieval heads: 128
Streaming heads: 128
Streaming fraction: 50.0%

Per-layer breakdown (R=retrieval, S=streaming):
  Layer  0: [SSRSSRSS] (6/8 streaming)
  Layer  1: [RSSRSSSR] (5/8 streaming)
  Layer  2: [SRSRRRSS] (4/8 streaming)
  Layer  3: [SSSSSRSS] (7/8 streaming)
  Layer  4: [SSRSRRRS] (4/8 streaming)
  Layer  5: [SSRSSSRS] (6/8 streaming)
  Layer  6: [SSSSSRRS] (6/8 streaming)
  Layer  7: [RSSRRRRR] (2/8 streaming)
  Layer  8: [RRRSS

## Cell 5: Apply StreamingAttention Monkey Patch

In [5]:
from streaming_attention.hybrid_attention import patch_model_for_streaming_attention, StreamingAttentionConfig

state_config = StreamingAttentionConfig(
    decay_init=0.99,
    learnable_decay=False,  # Fixed decay for zero-shot
    chunk_size=64,
)

print("Applying StreamingAttention monkey patch...")
print(f"  Decay factor: {state_config.decay_init}")
print(f"  Chunk size: {state_config.chunk_size}")

model, state_cache = patch_model_for_streaming_attention(
    model=model,
    head_classification=head_classification,
    config=state_config,
)

print(f"\nPatch applied successfully!")
print(f"State modules: {len(model._streaming_attn_modules)}")
if torch.cuda.is_available():
    print(f"GPU memory after patch: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Applying StreamingAttention monkey patch...
  Decay factor: 0.99
  Chunk size: 64

Patch applied successfully!
State modules: 128
GPU memory after patch: 16.07 GB


## Cell 6: Zero-Shot Perplexity (State-Converted Streaming Heads)

In [7]:
print("="*60)
print("KV2STATE: Zero-shot perplexity with state-converted streaming heads")
print("="*60)

state_cache.reset()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

streaming_attention_result = evaluate_perplexity(
    model=model,
    tokenizer=tokenizer,
    dataset_name='wikitext',
    dataset_config='wikitext-2-raw-v1',
    split='test',
    max_length=2048,
    stride=512,
    device=DEVICE,
)

ratio = streaming_attention_result['perplexity'] / baseline_result['perplexity']

print(f"\n{'='*60}")
print(f"RESULTS COMPARISON")
print(f"{'='*60}")
print(f"Baseline perplexity:  {baseline_result['perplexity']:.2f}")
print(f"StreamingAttention perplexity:  {streaming_attention_result['perplexity']:.2f}")
print(f"Degradation ratio:    {ratio:.2f}x")
if torch.cuda.is_available():
    print(f"Peak GPU memory:      {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print()

if ratio < 3.0:
    print(">>> GO: Perplexity degradation < 3x. Proceed to calibration (Sprint 2).")
elif ratio < 10.0:
    print(">>> MARGINAL: Perplexity degradation between 3-10x.")
    print(">>> Consider adjusting decay factor, trying GLA/DeltaNet, or reducing conversion fraction.")
else:
    print(">>> STOP: Perplexity degradation > 10x. Fundamental issue with approach.")

KV2STATE: Zero-shot perplexity with state-converted streaming heads


Perplexity (wikitext):   1%|          | 3/565 [01:08<3:34:32, 22.90s/it, ppl=5464.86, tokens=3072]


KeyboardInterrupt: 

## Cell 7: Streaming Fraction Sweep

Sweep over different streaming fractions to find the quality-efficiency frontier.
Each fraction requires reloading the model (patching is in-place).

In [8]:
import gc
from streaming_attention.hybrid_attention import patch_model_for_streaming_attention, StreamingAttentionConfig
from streaming_attention.head_classifier import load_duo_attention_patterns
from streaming_attention.eval_perplexity import evaluate_perplexity

fractions = [0.1, 0.2, 0.3, 0.5]
fraction_results = {}

DUO_DIR = 'duo-attention/attn_patterns/Meta-Llama-3.1-8B-Instruct'

print(f"Sweeping streaming fractions: {fractions}")
print(f"Baseline perplexity: {baseline_result['perplexity']:.2f}")
print()

for frac in fractions:
    print(f"{'='*60}")
    print(f"Testing streaming fraction = {frac:.0%}")
    print(f"{'='*60}")

    # Reload fresh model (patching is in-place, can't undo)
    model_sweep = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        attn_implementation='eager',
    )
    model_sweep.eval()

    # Classify heads at this fraction
    hc = load_duo_attention_patterns(DUO_DIR, sparsity=frac)
    print(f"  Retrieval: {hc.num_retrieval}, Streaming: {hc.num_streaming}")

    # Patch
    cfg = StreamingAttentionConfig(decay_init=0.99, learnable_decay=False, chunk_size=64)
    model_sweep, sc = patch_model_for_streaming_attention(model_sweep, hc, cfg)
    sc.reset()

    # Evaluate
    result = evaluate_perplexity(
        model=model_sweep,
        tokenizer=tokenizer,
        dataset_name='wikitext',
        dataset_config='wikitext-2-raw-v1',
        split='test',
        max_length=2048,
        stride=512,
        device=DEVICE,
    )

    ratio = result['perplexity'] / baseline_result['perplexity']
    fraction_results[frac] = result
    print(f"  Perplexity: {result['perplexity']:.2f} ({ratio:.2f}x baseline)")
    print()

    # Free memory
    del model_sweep, sc
    gc.collect()
    torch.cuda.empty_cache()

# Summary table
print("\n" + "="*60)
print(f"{'Fraction':<12} {'Streaming':<12} {'Perplexity':<15} {'Ratio':<10} {'Status':<15}")
print("="*60)
for frac, result in fraction_results.items():
    r = result['perplexity'] / baseline_result['perplexity']
    n_stream = int(256 * frac)
    status = 'GO' if r < 3.0 else ('MARGINAL' if r < 10.0 else 'STOP')
    print(f"{frac:<12.0%} {n_stream:<12} {result['perplexity']:<15.2f} {r:<10.2f} {status:<15}")
print(f"{'Baseline':<12} {'0':<12} {baseline_result['perplexity']:<15.2f} {'1.00':<10} {'Reference':<15}")

Sweeping streaming fractions: [0.1, 0.2, 0.3, 0.5]
Baseline perplexity: 6.43

Testing streaming fraction = 10%


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Retrieval: 229, Streaming: 27


Perplexity (wikitext):   8%|▊         | 48/565 [03:24<36:45,  4.27s/it, ppl=1256.39, tokens=26112]


KeyboardInterrupt: 

## Cell 8: Visualize Head Classification & Entropy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Head classification heatmap
ax1 = axes[0]
mask_np = head_classification.mask.cpu().numpy().astype(float)
im = ax1.imshow(mask_np, aspect='auto', cmap='RdYlGn', interpolation='nearest')
ax1.set_xlabel('KV Head Index')
ax1.set_ylabel('Layer Index')
ax1.set_title('Head Classification\n(Green=Retrieval, Red=Streaming)')
ax1.set_xticks(range(mask_np.shape[1]))
plt.colorbar(im, ax=ax1, label='1=Retrieval, 0=Streaming')

# Plot 2: Raw scores heatmap
ax2 = axes[1]
if head_classification.raw_scores is not None:
    scores_np = head_classification.raw_scores.cpu().numpy()
    im2 = ax2.imshow(scores_np, aspect='auto', cmap='viridis', interpolation='nearest')
    ax2.set_xlabel('KV Head Index')
    ax2.set_ylabel('Layer Index')
    ax2.set_title(f'Raw Scores ({classification_method})')
    ax2.set_xticks(range(scores_np.shape[1]))
    plt.colorbar(im2, ax=ax2, label='Score')
else:
    ax2.text(0.5, 0.5, 'No raw scores', ha='center', va='center', transform=ax2.transAxes)

# Plot 3: Score distribution
ax3 = axes[2]
if head_classification.raw_scores is not None:
    flat_scores = head_classification.raw_scores.cpu().numpy().flatten()
    ax3.hist(flat_scores, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax3.axvline(x=0.5, color='red', linestyle='--', label='Default threshold')
    ax3.set_xlabel('Score')
    ax3.set_ylabel('Count')
    ax3.set_title('Score Distribution\n(Bimodal = good separation)')
    ax3.legend()
else:
    ax3.text(0.5, 0.5, 'No raw scores', ha='center', va='center', transform=ax3.transAxes)

plt.tight_layout()
plt.savefig('head_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: head_classification.png")

# Plot 4: Streaming fraction sweep results
if fraction_results:
    fig2, (ax4, ax5) = plt.subplots(1, 2, figsize=(12, 5))

    fracs = list(fraction_results.keys())
    ppls = [r['perplexity'] for r in fraction_results.values()]
    ratios = [r['perplexity'] / baseline_result['perplexity'] for r in fraction_results.values()]

    ax4.plot([f*100 for f in fracs], ppls, 'bo-', markersize=8, linewidth=2)
    ax4.axhline(y=baseline_result['perplexity'], color='green', linestyle='--', label='Baseline')
    ax4.axhline(y=3*baseline_result['perplexity'], color='orange', linestyle='--', label='3x (GO)')
    ax4.set_xlabel('Streaming Fraction (%)')
    ax4.set_ylabel('Perplexity')
    ax4.set_title('Perplexity vs Streaming Fraction')
    ax4.legend()

    colors = ['green' if r < 3 else 'orange' if r < 10 else 'red' for r in ratios]
    ax5.bar(range(len(fracs)), ratios, color=colors)
    ax5.set_xticks(range(len(fracs)))
    ax5.set_xticklabels([f'{f:.0%}' for f in fracs])
    ax5.axhline(y=3.0, color='orange', linestyle='--', label='GO threshold (3x)')
    ax5.axhline(y=10.0, color='red', linestyle='--', label='STOP threshold (10x)')
    ax5.set_xlabel('Streaming Fraction')
    ax5.set_ylabel('Perplexity Ratio (vs baseline)')
    ax5.set_title('Degradation vs Streaming Fraction')
    ax5.legend()

    plt.tight_layout()
    plt.savefig('fraction_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: fraction_sweep.png")

## Summary & Next Steps

### Results
| Metric | Value |
|--------|-------|
| Baseline PPL | _(fill in)_ |
| StreamingAttention PPL (best decay) | _(fill in)_ |
| Degradation ratio | _(fill in)_ |
| Best decay factor | _(fill in)_ |

### GO/NO-GO Decision
- [ ] **GO** (< 3x): Proceed to Sprint 2 — calibration training
- [ ] **MARGINAL** (3-10x): Adjust decay, try GLA/DeltaNet, reduce streaming fraction
- [ ] **STOP** (> 10x): Re-evaluate approach fundamentally

### Sprint 2 Preview
If GO:
1. **Stage 1**: Per-head MSE alignment — learn optimal decay per head
2. **Stage 2**: End-to-end LoRA fine-tuning with cross-entropy loss
3. **Target**: Perplexity within 5% of baseline